In [ ]:
pip install ultralytics roboflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 70.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 84.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 120.2 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.12.0.88
    Uninstalling opencv-python-headless-4.12.0.88:
      Successfully uninstalled opencv-python-headless-4.12.0.88
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11


In [ ]:
from ultralytics import YOLO
import itertools
import pandas as pd
import os
import torch
from roboflow import Roboflow
import zipfile
import shutil
import os
from google.colab import files

In [ ]:
rf = Roboflow(api_key="a1uozLDJeFIwfWwaSoy6")
project = rf.workspace("s-gbust").project("armed-person-recognition-gohff")
version = project.version(4)
dataset = version.download("yolov8")


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Armed-Person-Recognition-4 in yolov8:: 100%|██████████| 17366/17366 [00:02<00:00, 8478.13it/s] 


In [ ]:
modelFolder = "/content/model"
os.makedirs(modelFolder, exist_ok=True)
modelResult = "/content/Result.txt"
result = open(modelResult, "w")
result.close()

In [ ]:
# Melatih model YOLOv8
def train_yolo(modelName, modelPathSave):
    # Path ke file konfigurasi dataset (.yaml)
    dataset_yaml = '/content/Armed-Person-Recognition-4/data.yaml'

    # Inisialisasi dan training model YOLOv8
    model = YOLO(modelName)  # Menggunakan model YOLOv8 nano (versi ringan)

    model.train(
        data=dataset_yaml,  # Path ke dataset
        epochs=100,          # Epochs = 150
        imgsz=640,           # Ukuran gambar (default 640x640)
        batch=128,            # Batch size = 32
        lr0=0.001,           # Learning rate = 0.001
        weight_decay=0.0005, # Weight decay = 0.0005
        momentum=0.937,      # Momentum = 0.937
        iou=0.5,      # IoU threshold = 0.5
        conf=0.25         # Jumlah workers (misalnya 4)
    )

    # Menyimpan model setelah pelatihan
    model.save(modelPathSave)

# Fungsi untuk mengevaluasi model pada data validasi dan tes
def evaluate_model(model_path, detail):
    # Memuat model yang sudah dilatih
    model = YOLO(model_path)
    dataset_yaml = "/kaggle/working/Armed-Person-Recognition-13/data.yaml"
    # Evaluasi model pada dataset validasi
    results_val = model.val(data=dataset_yaml, split='val')

    # Evaluasi model pada dataset tes
    results_test = model.val(data=dataset_yaml, split='test')


    with open(modelResult, 'a') as file:
        file.write(f"\n {'-'*10} {detail} {'-'*10}")
        file.write(f"\n {'-'*5} val {'-'*5}")
        file.write(f"{results_val}")
        file.write(f"{'-'*30}")
        file.write(f"{results_test}")


In [ ]:
train_yolo("yolov8s.pt", f"{modelFolder}/yolov9sVer0.pt")


Ultralytics 8.3.233 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (NVIDIA A100-SXM4-40GB, 40507MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=128, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=0.25, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Armed-Person-Recognition-4/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.5, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, pe

In [ ]:
folder_path = "/content/runs" # This is the folder name from the output of the training run
zip_path = "/content/baseline_yolov8.zip"

if os.path.exists(folder_path):
    shutil.make_archive(zip_path.replace(".zip", ""), 'zip', folder_path)
    print(f"Folder '{folder_path}' zipped to '{zip_path}'")
    files.download(zip_path)
else:
    print(f"Folder '{folder_path}' not found.")

Folder '/content/runs' zipped to '/content/baseline_yolov8.zip'


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>